# SSDL SatPass Notification
Written by Kiyoaki Okudaira<br>
*Kyushu University Hanada Lab / University of Washington / IAU CPS SatHub<br>
(okudaira.kiyoaki.528@s.kyushu-u.ac.jp or kiyoaki@uw.edu)<br>
<br>
This app notifies bright artificial objects expected in 10 days. Data provided by heavens-above.com and SatPhotometry Library.<br>
<br>
**History**<br>
coding 2026-02-24 : 1st coding<br>
<br>
(c) 2026 Kiyoaki Okudaira - Kyushu University Hanada Lab (SSDL) / University of Washington / IAU CPS SatHub

### Parameters
**Target**<br>
In UTC

In [1]:
from input.satlist.BRIGHT_LEO import *

**Heavens-Above settings**

In [2]:
min_alt      = 30   # minimum altitude of objects [deg] | int or float
min_duration = 60  # minimum duration of objects [sec] | int or float
time_window  = "evening"    # morning, evening or all   | str or bool

**Slack API settings**

In [ ]:
token       = ""        # Slack App API token | str
channel_id  = ""        # Channel ID | str
notify_type = "bydate"  # Notification type; "bydate" or "bysat" | str

### Import and initial settings
**PATH settings**

In [4]:
base_PATH = "/Users/kiyoaki/VScode/satphotometry_package/"
output_PATH = base_PATH + "output/heavens-above"
input_PATH = base_PATH + "input/heavens-above"

**Standard libraries**

In [5]:
import numpy as np
from __future__ import annotations
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from time import sleep
import os, requests, json

from astropy.time import Time, TimeDelta
import astropy.units as u
from astroplan import Observer
from astropy.table import vstack

**Satphotometry library**

In [6]:
from satphotometry import heavens_above,gettle

**Observatory setting**

In [7]:
from input.obs_site.KUPT import *

obs_obj = Observer(
    longitude = obs_gd_lon_deg * u.deg,
    latitude = obs_gd_lat_deg * u.deg,
    timezone = obs_timezone,
    name = obs_name
)

now_local = datetime.now(ZoneInfo(obs_timezone))
offset = now_local.utcoffset()

lst_h = offset.total_seconds() // 3600
lst_m = (offset.total_seconds() % 3600) // 60


**Global constants**

In [8]:
WEEKDAY_JP = ["月", "火", "水", "木", "金", "土", "日"]
HEAVENS_ABOVE_URL = "https://www.heavens-above.com/"

### Heavens-Above
**Get pass Summary**<br>
Get pass Summary from www.heavens-above.com/PassSummary.aspx

In [9]:
pass_table = None
for norad_id in norad_ids:
    _,tle_result = gettle.celes_trak.get_latest_TLE(norad_id)
    with open(f"{base_PATH}tmp/heavens-above/tle.txt","w") as f:
        f.write(tle_result)
    tle_dict = gettle.parse.parse_tles_file(f"{base_PATH}tmp/heavens-above/tle.txt")
    satname = tle_dict[str(norad_id)][0]['name'].rstrip()

    query_result = heavens_above.get_pass_summary(norad_id,obs_gd_lon_deg,obs_gd_lat_deg,obs_gd_height,"UCT")
    sat_pass_table = heavens_above.parse_summary2table(query_result,satname)
    if pass_table is None:
        pass_table = sat_pass_table
    else:
        pass_table = vstack([pass_table, sat_pass_table])
    sleep(0.25)
pass_table.write(f"{output_PATH}/pass_summary.csv",overwrite=True)

**Timewindow**<br>
Morning / Evening observation

In [10]:
time_windows = []
dates = []
for row in pass_table:
    obs_start = Time(row["start_utc"]).mjd
    obs_noon = obs_obj.noon(Time(row["start_utc"]), which = "previous")
    sun_horizon = obs_obj.tonight(obs_noon, horizon = 0 * u.deg)
    sunset_lst  = sun_horizon[0].mjd
    sunrise_lst = sun_horizon[1].mjd
    if abs(obs_start-sunset_lst) < abs(obs_start-sunrise_lst):
        time_windows.append("evening")
    else:
        time_windows.append("morning")
    dates.append((Time(row["max_utc"]) + TimeDelta(lst_h*u.hour + lst_m*u.minute)).isot[0:10])
pass_table["date"] = dates
pass_table["time_window"] = time_windows

### iCalendar

In [11]:
def _ics_dt(dt: datetime) -> str:
    return dt.astimezone(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

def _ics_lst_dt(dt: datetime) -> str:
    return dt.strftime("%Y%m%dT%H%M%S")

def _ics_escape(text: str) -> str:
    text = str(text)
    return (
        text.replace("\\", "\\\\")
            .replace(";", r"\;")
            .replace(",", r"\,")
            .replace("\r\n", r"\n")
            .replace("\n", r"\n")
    )

def _ics_escape_uri(uri: str) -> str:
    return str(uri).replace("\\", "\\\\")

def _fold_ics_line(line: str, limit: int = 75) -> str:
    if len(line) <= limit:
        return line
    out = []
    while len(line) > limit:
        out.append(line[:limit])
        line = " " + line[limit:]
    out.append(line)
    return "\r\n".join(out)


def write_passes_to_ics(pass_table, out_path, calendar_name: str = "Satellite Passes") -> str:
    out_path
    now_utc = datetime.now(timezone.utc)

    lines = [
        "BEGIN:VCALENDAR",
        "VERSION:2.0",
        "PRODID:-//SatPhotometry//SatPass//EN",
        "CALSCALE:GREGORIAN",
        "METHOD:PUBLISH",
        f"X-WR-CALNAME:{_ics_escape(calendar_name)}",
    ]

    # Astropy Table rows can be iterated directly
    for row in pass_table:
        satid = row["satid"]
        satname = row["satname"]

        event_url = f"{HEAVENS_ABOVE_URL}{row["detail_url"]}"

        start_lst_obj = (Time(row["start_utc"]) + TimeDelta(lst_h*u.hour + lst_m*u.minute))
        start_lst_time = start_lst_obj.isot[11:19]
        max_lst_time = (Time(row["max_utc"]) + TimeDelta(lst_h*u.hour + lst_m*u.minute)).isot[11:19]
        end_lst_obj = (Time(row["end_utc"]) + TimeDelta(lst_h*u.hour + lst_m*u.minute))
        end_lst_time = (Time(row["end_utc"]) + TimeDelta(lst_h*u.hour + lst_m*u.minute)).isot[11:19]

        start_lst_dt = start_lst_obj.to_datetime()
        end_lst_dt = end_lst_obj.to_datetime()

        obs_noon = obs_obj.noon(Time(row["start_utc"]), which = "previous")
        sun_horizon = obs_obj.tonight(obs_noon, horizon = 0 * u.deg)
        astro_twilight = obs_obj.tonight(obs_noon, horizon = -18 * u.deg)
        sunset_lst  = (sun_horizon[0]+ TimeDelta(lst_h*u.hour + lst_m*u.minute)).isot[11:16]
        sunrise_lst = (sun_horizon[1]+ TimeDelta(lst_h*u.hour + lst_m*u.minute)).isot[11:16]
        astro_dusk_lst = (astro_twilight[0]+ TimeDelta(lst_h*u.hour + lst_m*u.minute)).isot[11:16]
        astro_dawn_lst = (astro_twilight[1]+ TimeDelta(lst_h*u.hour + lst_m*u.minute)).isot[11:16]

        # Summary (event title)
        summary = f"{satname}"

        # Description (details)
        desc = (
            f"================================\n"
            f"{satname} | NORAD ID {satid}\n"
            f"================================\n"
            f"Mag : {row['mag']}\n"
            f"Duration : {row['duration']//60} min {row['duration']%60} sec\n"
            f"Pass start : {start_lst_time} (el={row['start_alt']} deg / {row['start_az']})\n"
            f"Highest : {max_lst_time} (el={row['max_alt']} deg / {row['max_az']})\n"
            f"Pass end : {end_lst_time} (el={row['end_alt']} deg / {row['end_az']})\n"
            f"----------------------------------------\n"
            f"Sunset : {sunset_lst}\n"
            f"Astronomical dusk : {astro_dusk_lst}\n"
            f"Astronomical dawn : {astro_dawn_lst}\n"
            f"Sunrise : {sunrise_lst}\n"
            f"----------------------------------------\n"
            f"Data Provided by Heavens-Above\n"
            f"Created / updated at {Time.now().isot[0:19]}\n"
            f"================================\n"
            f"SSDL SatPass Notification System\n"
            f"with SatPhotometry Library\n"
            f"(c) 2026 Kiyoaki Okudaira - Kyushu University\n"
            f"================================"
        )
        uid = f"{satid}.{row["mjd"]:.1f}@SatPass"

        event_lines = [
            "BEGIN:VEVENT",
            f"UID:{uid}",
            f"DTSTAMP:{_ics_dt(now_utc)}",
            f"DTSTART;TZID={obs_timezone}:{_ics_lst_dt(start_lst_dt)}",
            f"DTEND;TZID={obs_timezone}:{_ics_lst_dt(end_lst_dt)}",
            f"SUMMARY:{_ics_escape(summary)}",
            f"LOCATION:{obs_name}",
            f"GEO:{obs_gd_lat_deg:.6f};{obs_gd_lon_deg:.6f}",
            f"X-APPLE-STRUCTURED-LOCATION;VALUE=URI;X-APPLE-RADIUS=72;X-TITLE={obs_name}:geo:{obs_gd_lat_deg:.6f},{obs_gd_lon_deg:.6f}",
            f"URL:{_ics_escape_uri(event_url)}",
            f"DESCRIPTION:{_ics_escape(desc)}",
            "END:VEVENT",
        ]

        # Fold long lines
        for el in event_lines:
            lines.append(_fold_ics_line(el))

    lines.append("END:VCALENDAR")

    ics_text = "\r\n".join(lines) + "\r\n"
    with open(out_path, "w", encoding="utf-8", newline="") as f:
        f.write(ics_text)

    return out_path

In [ ]:
if time_window == "evening" or time_window == "morning":
    good_pass_table = pass_table[(pass_table["max_alt"] > min_alt) & (pass_table["duration"] > min_duration) & (pass_table["visible"] == True) & (pass_table["time_window"] == time_window)]
else:
    good_pass_table = pass_table[(pass_table["max_alt"] > min_alt) & (pass_table["duration"] > min_duration) & (pass_table["visible"] == True)]

good_pass_table = good_pass_table.group_by("satname")
out_file = write_passes_to_ics(good_pass_table, out_path=f"{output_PATH}/SatPass.ics")

### Slack Notification

**Construct contents**<br>
By satellites

In [ ]:
if notify_type == "bysat":
    lines = []
    lines.append(f"*🛰️ 注目すべき人工天体の上空通過予測 (試験運用)*")
    lines.append(f"直近10日間の注目すべき人工天体の上空通過予測をお知らせします．")
    lines.append(f"(Filter : alt > {min_alt} deg & duration > {min_duration} sec & time window = {time_window})")
    lines.append("")
    
    for group in pass_table.groups:
        satname = group[0]["satname"]
        norad_id = group[0]["satid"]

        if len(pass_table) > 0:
            lines.append(f"*{satname} (NORAD ID {norad_id})* は直近10日間で{len(group)}件の観測可能な上空通過が予測されています．")
        else:
            lines.append(f"*{satname} (NORAD ID {norad_id})* は直近10日間に観測可能な上空通過がありません．")
        
        if time_window == "evening" or time_window == "morning":
            good_condition = (group["max_alt"] > 30) & (group["duration"] > 120) & (group["visible"] == True) & (pass_table["time_window"] == time_window)
        else:
            good_condition = (group["max_alt"] > 30) & (group["duration"] > 120) & (group["visible"] == True)

        if np.sum(good_condition) > 0:
            lines.append(f"良い観測条件の上空通過({np.sum(good_condition)}件)は以下の通りです．")
            lines.append("")

            table_lines = []
            table_lines.append("観測日            Pass Start           Highest              End")

            for row in group[good_condition]:
                start_lst_obj = (Time(row["start_utc"]) + TimeDelta(lst_h*u.hour + lst_m*u.minute))
                start_lst_weekday = WEEKDAY_JP[start_lst_obj.to_datetime().weekday()]
                start_lst_date = start_lst_obj.isot[0:10]
                start_lst_time = start_lst_obj.isot[11:19]

                max_lst_time = (Time(row["max_utc"]) + TimeDelta(lst_h*u.hour + lst_m*u.minute)).isot[11:19]
                end_lst_time = (Time(row["end_utc"]) + TimeDelta(lst_h*u.hour + lst_m*u.minute)).isot[11:19]

                table_lines.append(
                    f"{start_lst_date} {start_lst_weekday}曜日 "
                    + f"{start_lst_time} ({row['start_alt']:.0f}deg {row['start_az']})".ljust(21)
                    + f"{max_lst_time} ({row['max_alt']:.0f}deg {row['max_az']})".ljust(21)
                    + f"{end_lst_time} ({row['end_alt']:.0f}deg {row['end_az']})".ljust(21)
                )

            # Code block
            lines.append("```" + "\n".join(table_lines) + "```")
        else:
            lines.append("良い観測条件の上空通過はありません．")
        lines.append("")

    lines.append("This message is automatically sent by SSDL SatPass Notification System")
    lines.append("Data Provided by <https://www.heavens-above.com|Heavens-Above> / SatPhotometry Library")

By date

In [ ]:
if notify_type == "bydate":
    if time_window == "evening" or time_window == "morning":
        good_pass_table = pass_table[(pass_table["max_alt"] > min_alt) & (pass_table["duration"] > min_duration) & (pass_table["visible"] == True) & (pass_table["time_window"] == time_window)]
    else:
        good_pass_table = pass_table[(pass_table["max_alt"] > min_alt) & (pass_table["duration"] > min_duration) & (pass_table["visible"] == True)]
    good_pass_table.sort("start_utc")
    good_pass_table = good_pass_table.group_by("date")

    lines = []
    lines.append(f"*🛰️ 注目すべき人工天体の上空通過予測 (試験運用)*")
    lines.append(f"直近10日間の注目すべき人工天体の上空通過予測をお知らせします．")
    lines.append(f"(Filter : alt > {min_alt} deg & duration > {min_duration} sec & time window = {time_window})")
    lines.append("")

    if len(good_pass_table) > 0:
        for group in good_pass_table.groups:
            date = group[0]["date"]
            start_lst_obj = (Time(group[0]["start_utc"]) + TimeDelta(lst_h*u.hour + lst_m*u.minute))
            start_lst_weekday = WEEKDAY_JP[start_lst_obj.to_datetime().weekday()]

            lines.append(f"*{date[0:4]}年{date[5:7]}月{date[8:10]}日*")
            lines.append(f"注目すべき衛星の良い観測条件の上空通過が{len(group)}件予測されています．")

            lines.append("")

            table_lines = []
            table_lines.append("Satellite            Pass Start           Highest              End")
            for row in group:
                start_lst_obj = (Time(row["start_utc"]) + TimeDelta(lst_h*u.hour + lst_m*u.minute))
                start_lst_weekday = WEEKDAY_JP[start_lst_obj.to_datetime().weekday()]
                start_lst_date = start_lst_obj.isot[0:10]
                start_lst_time = start_lst_obj.isot[11:19]

                max_lst_time = (Time(row["max_utc"]) + TimeDelta(lst_h*u.hour + lst_m*u.minute)).isot[11:19]
                end_lst_time = (Time(row["end_utc"]) + TimeDelta(lst_h*u.hour + lst_m*u.minute)).isot[11:19]

                table_lines.append(
                    f"{row["satname"]}".ljust(21) 
                    + f"{start_lst_time} ({row['start_alt']:.0f}deg {row['start_az']})".ljust(21)
                    + f"{max_lst_time} ({row['max_alt']:.0f}deg {row['max_az']})".ljust(21)
                    + f"{end_lst_time} ({row['end_alt']:.0f}deg {row['end_az']})".ljust(21)
                )

            # Code block
            lines.append("```" + "\n".join(table_lines) + "```")

            lines.append("")
    else:
        lines.append("直近10日間に注目すべき人工天体の容易観測条件での上空通過はありません．")
        lines.append("")

    lines.append("This message is automatically sent by SSDL SatPass Notification System")
    lines.append("Data Provided by <https://www.heavens-above.com|Heavens-Above> / SatPhotometry Library")

**Preview**

In [15]:
for f in lines:
    print(f)

<!channel>*🛰️ 注目すべき人工天体の上空通過予測 (試験運用)*
直近10日間の注目すべき人工天体の上空通過予測をお知らせします．
(Filter : alt > 30 deg & duration > 60 sec & time window = evening)

*2026年03月04日*
注目すべき衛星の良い観測条件の上空通過が3件予測されています．

```Satellite            Pass Start           Highest              End
SPACEMOBILE-002      19:36:33 (10deg SSW) 19:40:10 (39deg SE)  19:40:10 (39deg SE)  
SPACEMOBILE-001      20:00:41 (10deg SW)  20:03:42 (51deg SSW) 20:03:42 (51deg SSW) 
SPACEMOBILE-005      20:04:02 (10deg SW)  20:06:55 (48deg SSW) 20:06:55 (48deg SSW) ```

*2026年03月05日*
注目すべき衛星の良い観測条件の上空通過が4件予測されています．

```Satellite            Pass Start           Highest              End
SPACEMOBILE-002      19:15:27 (10deg SSW) 19:19:05 (40deg SE)  19:21:03 (22deg ENE) 
SPACEMOBILE-001      19:39:34 (10deg SW)  19:43:26 (75deg SE)  19:44:32 (44deg ENE) 
SPACEMOBILE-005      19:43:12 (10deg SW)  19:47:05 (81deg SE)  19:48:02 (50deg NE)  
SPACEMOBILE-003      20:05:58 (10deg SW)  20:09:47 (59deg NW)  20:09:50 (59deg NW)  ```

*2026年03月06日*
注目すべき衛星の

**Send notification**

In [ ]:
content = "\n".join(lines)

# upload file path
file_path = f"{output_PATH}/SatPass.ics"

# comment
title = os.path.basename(file_path)
initial_comment = "pass summary csv"

def slack_api_post(url: str, token: str, data=None, files=None, timeout=60):
    r = requests.post(
        url,
        headers={"Authorization": f"Bearer {token}"},
        data=data,
        files=files,
        timeout=timeout,
    )
    r.raise_for_status()
    payload = r.json()
    if not payload.get("ok", False):
        raise RuntimeError(f"Slack API error: {payload}")
    return payload

# retrieve file id and url
file_size = os.path.getsize(file_path)
filename = os.path.basename(file_path)

get_url_payload = slack_api_post(
    "https://slack.com/api/files.getUploadURLExternal",
    token=token,
    data={
        "filename": filename,
        "length": str(file_size),  # bytes
    },
)
upload_url = get_url_payload["upload_url"]
file_id = get_url_payload["file_id"]

# upload file
with open(file_path, "rb") as f:
    upload_resp = requests.post(
        upload_url,
        headers={"Content-Type": "application/octet-stream"},
        data=f,
        timeout=300,
    )
upload_resp.raise_for_status()

# confirm file share
complete_payload = slack_api_post(
    "https://slack.com/api/files.completeUploadExternal",
    token=token,
    data={
        "files": json.dumps([{"id": file_id, "title": title}]),
        "channel_id": channel_id,
        "initial_comment": content,
    },
)

print("Uploaded OK")
print(json.dumps(complete_payload, indent=2, ensure_ascii=False))

Uploaded OK
{
  "ok": true,
  "files": [
    {
      "id": "F0AH2DJHG92",
      "created": 1771984552,
      "timestamp": 1771984552,
      "name": "pass_events.ics",
      "title": "pass_events.ics",
      "mimetype": "",
      "filetype": "",
      "pretty_type": "",
      "user": "U0AHCDDE57S",
      "user_team": "T01UCT7RFN3",
      "editable": false,
      "size": 15400,
      "mode": "hosted",
      "is_external": false,
      "external_type": "",
      "is_public": false,
      "public_url_shared": false,
      "display_as_bot": false,
      "username": "",
      "url_private": "https://files.slack.com/files-pri/T01UCT7RFN3-F0AH2DJHG92/pass_events.ics",
      "url_private_download": "https://files.slack.com/files-pri/T01UCT7RFN3-F0AH2DJHG92/download/pass_events.ics",
      "media_display_type": "unknown",
      "permalink": "https://ssdlab-ku.slack.com/files/U0AHCDDE57S/F0AH2DJHG92/pass_events.ics",
      "permalink_public": "https://slack-files.com/T01UCT7RFN3-F0AH2DJHG92-cdb28